<h2 style="color:#FF7A70;">Clase Proyecto Final: Global Economic Analysis Report 2025</h2>

<p><strong>Curso:</strong> Lenguaje y Programación II</p>

<p><strong>Integrantes del grupo:</strong></p>
<ul>
  <li>Castillo Flores, Hellary Mayte — <em>20240699</em></li>
  <li>Cruz Lozano, Gianella Alejandra — <em>20240705</em></li>
  <li>Tineo Balcázar, Daniela Rosa — <em>20231510</em></li>
</ul>

<h2 style="color:#FF7A70;">Objetivo:</h2>
<p>
Analizar el <strong>Top 10 de países por PIB Nominal</strong> utilizando datos oficiales
del <strong>Banco Mundial</strong>, e incorporar una métrica comparativa adicional llamada
<strong>Capacidad BTC</strong>, basada en el precio actual de Bitcoin, con el fin de
ofrecer una visión económica alternativa y moderna.
</p>

<h2 style="color:#FF7A70;">Fuentes de datos:</h2>
<ul>
  <li><strong>World Bank API</strong> – Indicadores macroeconómicos</li>
  <li><strong>CoinGecko API</strong> – Precio actual de Bitcoin (USD)</li>
</ul>
<h2 style="color:#FF7A70;">Parte 1: Carga de datos y APIs</h2>
<p>En esta sección vamos a:</p>
<ol>
  <li>Descargar el precio de Bitcoin desde CoinGecko.</li>
  <li>Descargar indicadores económicos (PIB, PIB per cápita, crecimiento) desde World Bank.</li>
  <li>Obtener la población de cada país usando REST Countries (optimizado para rapidez).</li>
  <li>Calcular la Capacidad BTC y PIB per cápita real.</li>
</ol>





In [6]:
import pandas as pd
import requests
import wbgapi as wb
import warnings

warnings.filterwarnings("ignore")  # Ignorar advertencias

# -----------------------------
# Transformar valores numéricos en cadenas de texto
# -----------------------------
def format_money(val):
    if pd.isna(val): return "N/A"
    if val >= 1e12: return f"{val/1e12:,.2f} Trillion USD"
    if val >= 1e9: return f"{val/1e9:,.2f} Billion USD"
    return f"{val:,.2f} USD"

def format_btc(val):
    if pd.isna(val): return "N/A"
    return f"{val:,.0f} BTC"

def format_percent(val):
    if pd.isna(val): return "N/A"
    return f"{val:.2f} %"

def format_population(val):
    if pd.isna(val): return "N/A"
    return f"{val:,}"

# -----------------------------
# Cargar datos
# -----------------------------
# Precio BTC
btc_price = requests.get(
    "https://api.coingecko.com/api/v3/simple/price",
    params={"ids": "bitcoin", "vs_currencies": "usd"}
).json()["bitcoin"]["usd"]

# Indicadores económicos
indicadores = {
    "NY.GDP.MKTP.CD": "PIB_Nominal",
    "NY.GDP.PCAP.CD": "PIB_Per_Capita",
    "NY.GDP.MKTP.KD.ZG": "Crecimiento_PIB"
}

df = wb.data.DataFrame(indicadores.keys(), labels=True, mrnev=1).reset_index()
df = df.rename(columns={
    "Country": "Pais",
    "NY.GDP.MKTP.CD": "PIB_Nominal",
    "NY.GDP.PCAP.CD": "PIB_Per_Capita",
    "NY.GDP.MKTP.KD.ZG": "Crecimiento_PIB"
})

# Eliminamos agregados
paises_validos = [c["id"] for c in wb.economy.list() if not c["aggregate"]]
df = df[df["economy"].isin(paises_validos)]
df = df.dropna(subset=["PIB_Nominal"])

# -----------------------------
# Poblaciones (REST Countries)
# -----------------------------
resp = requests.get("https://restcountries.com/v3.1/all").json()
poblaciones_dict = {
    c["name"]["common"].lower(): c.get("population", None)
    for c in resp
    if isinstance(c, dict) and "name" in c and "common" in c["name"]
}

df["Poblacion"] = df["Pais"].str.lower().map(poblaciones_dict)
df["Poblacion"] = df["Poblacion"].fillna(df["PIB_Nominal"] / df["PIB_Per_Capita"])

# -----------------------------
# Calcular métricas
# -----------------------------
df["PIB_Per_Capita_Real"] = df["PIB_Nominal"] / df["Poblacion"]

df["Capacidad_BTC"] = df["PIB_Nominal"] / btc_price

df["Capacidad_BTC_per_capita"] = df["Capacidad_BTC"] / df["Poblacion"]

# -----------------------------
# Orden final
# -----------------------------
df = df.sort_values("PIB_Nominal", ascending=False).reset_index(drop=True)
df.index += 1

df.head()




,economy,Pais,PIB_Nominal,Crecimiento_PIB,PIB_Per_Capita,Poblacion,PIB_Per_Capita_Real,Capacidad_BTC,Capacidad_BTC_per_capita
1,USA,United States,2.875096e+13,2.793001,84534.040784,3.401110e+08,84534.040784,3.281735e+08,0.964901
2,CHN,China,1.874380e+13,4.977357,13303.148154,1.408975e+09,13303.148154,2.139484e+08,0.151847
3,DEU,Germany,4.685593e+12,-0.495852,56103.732318,8.351659e+07,56103.732318,5.348300e+07,0.640388
4,JPN,Japan,4.027598e+12,0.104309,32487.077805,1.239754e+08,32487.077805,4.597242e+07,0.370819
5,IND,India,3.909892e+12,6.494766,2694.737809,1.450936e+09,2694.737809,4.462888e+07,0.030759
